# Разведочный анализ данных сети АЗС

Ноутбук нужен для быстрой проверки структуры данных, целевых переменных, выбросов и причин, по которым модель может обучаться хуже на части признаков.

Основной источник данных: `_задание/5stations_data.csv`.


## 1. Загрузка данных

Сначала загружаем CSV, приводим время к `datetime` и смотрим размер таблицы.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

ROOT = Path('.').resolve().parent
df = pd.read_csv(ROOT / '_задание' / '5stations_data.csv', parse_dates=['timestamp'])
df.head()


## 2. Общая информация

Здесь смотрим размер, типы столбцов и базовую статистику. Это помогает понять, нет ли битых типов и насколько данные полные.


In [ ]:
print('shape =', df.shape)
print('stations =', df['station_id'].nunique())
print('period =', df['timestamp'].min(), '->', df['timestamp'].max())
print('rows per station =')
print(df.groupby('station_id').size())
print()
df.info()
print()
df.describe(include='all').transpose().head(30)


## 3. Пропуски и дубликаты

Нужно проверить, нет ли пустых значений и повторяющихся строк. Для временных рядов это особенно важно, потому что пропуски ломают историю наблюдений.


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('missing columns =', len(missing))
print(missing.head(20))
print()
print('duplicate rows =', df.duplicated().sum())
print('duplicate timestamp-station pairs =', df.duplicated(['timestamp', 'station_id']).sum())


## 4. Целевые переменные

Смотрим распределение двух целей: продаж топлива и выручки магазина. Если у цели много нулей или сильный перекос, модель обычно учится хуже.


In [ ]:
targets = ['total_fuel_sales', 'shop_total_revenue']
for col in targets:
    s = df[col]
    print(f'--- {col} ---')
    print('min =', s.min())
    print('max =', s.max())
    print('mean =', round(s.mean(), 3))
    print('std =', round(s.std(), 3))
    print('zero share =', round((s == 0).mean() * 100, 2), '%')
    print('quantiles =')
    print(s.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    print()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.ravel(), targets):
    sns.histplot(df[col], bins=50, kde=True, ax=ax, color='#2a6fdb')
    ax.set_title(f'Распределение: {col}')

for ax, col in zip(axes.ravel()[2:], targets):
    sns.boxplot(x=df[col], ax=ax, color='#f2b134')
    ax.set_title(f'Выбросы: {col}')
plt.tight_layout()
plt.show()


## 5. Выбросы

Для простой оценки выбросов используем IQR-правило. Это не строгий статистический тест, а удобный ориентир, где у данных слишком длинные хвосты.


In [ ]:
def iqr_outliers(series: pd.Series) -> int:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())

for col in ['total_fuel_sales', 'shop_total_revenue', 'total_traffic', 'temperature', 'precipitation_mm']:
    print(col, 'outliers =', iqr_outliers(df[col]))

numeric_cols = df.select_dtypes(include='number').columns
corr = df[numeric_cols].corr(numeric_only=True)[['total_fuel_sales', 'shop_total_revenue']].sort_values('total_fuel_sales', ascending=False)
print()
print(corr.head(20))


## 6. Поведение по станциям

Сравниваем средние продажи и долю нулей по каждой станции. Это помогает увидеть, не является ли одна из станций слишком слабой или слишком шумной для модели.


In [ ]:
station_summary = df.groupby('station_id').agg(
    fuel_mean=('total_fuel_sales', 'mean'),
    shop_mean=('shop_total_revenue', 'mean'),
    traffic_mean=('total_traffic', 'mean'),
    fuel_zero_share=('total_fuel_sales', lambda s: (s == 0).mean()),
    shop_zero_share=('shop_total_revenue', lambda s: (s == 0).mean()),
).reset_index()
print(station_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=station_summary, x='station_id', y='fuel_mean', ax=axes[0], color='#2a6fdb')
axes[0].set_title('Средние продажи топлива по станциям')
sns.barplot(data=station_summary, x='station_id', y='shop_mean', ax=axes[1], color='#f2b134')
axes[1].set_title('Средняя выручка магазина по станциям')
plt.tight_layout()
plt.show()


## 7. Почему модель могла обучиться хуже

Здесь собираем краткий вывод по данным: где сигнал сильный, а где много шума.


In [ ]:
print('Fuel zero share:', round((df['total_fuel_sales'] == 0).mean() * 100, 2), '%')
print('Shop zero share:', round((df['shop_total_revenue'] == 0).mean() * 100, 2), '%')
print('Average fuel:', round(df['total_fuel_sales'].mean(), 2))
print('Average shop:', round(df['shop_total_revenue'].mean(), 2))
print('Stations:', df['station_id'].nunique())
print('Main issue: shop revenue is sparse and has many zeros, so the model sees a weaker signal than for fuel.')


## 8. Финальный вывод

- В данных всего 5 АЗС, но много часов с нулевой выручкой магазина.
- Продажи топлива ведут себя стабильнее, поэтому модель по топливу учится лучше.
- Для магазина ряд более шумный и разреженный, из-за этого качество хуже.
- На защите можно сказать, что модель работает, но задача по магазину сложнее из-за структуры данных.
